# Eclat / Julia — Colab demo theo kế hoạch 3 module

**Template, chưa phải demo hoàn chỉnh.** A: thuật toán/performance; B: test,
Groceries, UCI Splice DNA và Colab. M1 28/10–03/11, M2 04/11–10/11,
M3 11/11–17/11/2026. Các mốc là lịch nội bộ.

Notebook dùng kernel Python để setup/gọi Julia bằng subprocess. Các cờ mặc định
tắt phần thuật toán/ứng dụng/benchmark chưa cài đặt; chạy các ô setup thành công
không có nghĩa đã đạt gate đồ án. B phải thay TODO và cold-run trước khi nộp.

In [ ]:
from pathlib import Path
import os
import sys
import hashlib
import re
import shutil
import subprocess
import urllib.request

REPO_URL = "https://github.com/phuctran33/-DM-Eclat.git"
REPO_REF = os.environ.get("ECLAT_REPO_REF", "feat/member-b-applications")
RUN_PREPARATION = True
JULIA_VERSION = "1.11.4"
RUN_MINER = False
RUN_SPMF = False
RUN_OPTIMIZED = False
RUN_APPLICATIONS = False
RUN_BENCHMARKS = False
FINAL_DELIVERY_VALIDATION = False
SEED = 42

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path("/content/eclat-member-b")
    if not (ROOT / "Project.toml").exists():
        if not REPO_URL.startswith("https://github.com/"):
            raise ValueError("Điền URL GitHub public thật của nhóm trước khi chạy Colab.")
        subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(ROOT)], check=True)
    if not shutil.which("julia"):
        archive = Path("/content/julia-runtime.tar.gz")
        runtime = Path("/content/julia-runtime")
        runtime.mkdir(exist_ok=True)
        urllib.request.urlretrieve(
            f"https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-{JULIA_VERSION}-linux-x86_64.tar.gz",
            archive,
        )
        checksum_url = f"https://julialang-s3.julialang.org/bin/checksums/julia-{JULIA_VERSION}.sha256"
        checksum_text = urllib.request.urlopen(checksum_url, timeout=30).read().decode("utf-8")
        filename = f"julia-{JULIA_VERSION}-linux-x86_64.tar.gz"
        expected = next(line.split()[0] for line in checksum_text.splitlines() if line.split()[-1].lstrip("*") == filename)
        assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected, "Julia archive checksum mismatch"
        subprocess.run(["tar", "-xzf", str(archive), "-C", str(runtime), "--strip-components=1"], check=True)
        os.environ["PATH"] = str(runtime / "bin") + os.pathsep + os.environ["PATH"]
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "Project.toml").exists()), None)
    if ROOT is None:
        raise RuntimeError("Mở notebook từ repo root hoặc notebooks/.")

os.chdir(ROOT)
os.environ.setdefault("JULIA_DEPOT_PATH", str(ROOT / ".tools/julia-depot"))
JULIA = os.environ.get("ECLAT_JULIA") or shutil.which("julia")
if not JULIA:
    candidates = sorted((ROOT / ".tools").glob("julia-*/bin/julia.exe"))
    JULIA = str(candidates[-1]) if candidates else None
if not JULIA:
    raise RuntimeError("Cài Julia 1.9+ và thêm vào PATH khi chạy local.")

def run_julia(*arguments, env=None):
    return subprocess.run(
        [JULIA, "--startup-file=no", f"--project={ROOT}", *map(str, arguments)],
        cwd=ROOT, env={**os.environ, **(env or {})}, check=True,
    )

version = subprocess.check_output([JULIA, "--version"], text=True)
match = re.search(r"(\d+)\.(\d+)", version)
assert match and tuple(map(int, match.groups())) >= (1, 9), "Julia 1.9+ required"
assert (ROOT / "scripts/prepare_applications.jl").is_file(), "Checkout phải có code B; chọn đúng nhánh đã publish hoặc mở source B local."
run_julia("--version")
run_julia("-e", "using Pkg; Pkg.instantiate()")
print("ROOT:", ROOT, "SEED:", SEED)

## B: preprocessing độc lập từ main
Đã kiểm chứng local: 206 test Julia + 4 Python, toàn bộ Groceries/Splice, 5 oracle SPMF thật.
Cell tiếp theo tải/kiểm source và chuẩn bị dữ liệu bằng Julia, không gọi miner A.
Colab cần nhánh B đã được push; chạy local có thể tự tìm Julia portable trong .tools.
Local cold-run và remote Colab là hai loại kiểm chứng khác nhau; chưa công bố remote Colab pass.


In [ ]:
if RUN_PREPARATION:
    import tempfile
    subprocess.run([sys.executable, "scripts/prepare_application_data.py", "--dataset", "all"], check=True)
    work_parent = ROOT / ".tools/notebook-runs"
    work_parent.mkdir(parents=True, exist_ok=True)
    B_OUTPUT = Path(tempfile.mkdtemp(prefix="member-b-", dir=work_parent))
    run_julia("scripts/prepare_applications.jl", "groceries", "data/applications/groceries.csv", B_OUTPUT / "groceries")
    run_julia("scripts/prepare_applications.jl", "splice", "data/applications/splice.data", B_OUTPUT / "splice")
    subprocess.run([sys.executable, "-B", "tests/verify_prepared_data.py", "--julia", JULIA,
                    "--report", str(B_OUTPUT / "preparation-report.json")], check=True)
    print("B preprocessing artifacts:", B_OUTPUT)
else:
    print("B preprocessing không chạy trong phiên này.")


## M1 — kiểm thử hạ tầng và CLI

Các test mặc định chỉ kiểm parser, threshold, output comparator; mining/SPMF còn
pending. Expected toy tự đếm độc lập ở 40%, không gọi là kết quả SPMF.

In [ ]:
run_julia("tests/runtests_b.jl")
subprocess.run([sys.executable, "-B", "tests/test_application_sources.py"], check=True)
run_julia("scripts/mine.jl", "--help")


## M1 — Eclat basic và SPMF oracle (B kiểm, A sửa)

Sau khi cài đặt basic, bật RUN_MINER. Tải JAR chính thức vào tools/spmf.jar,
ghi version/hash và tạo đủ năm file references ở minsup 40%, rồi bật RUN_SPMF.
Không dùng expected tự đếm để giả lập reference đã chạy SPMF. Đọc
tests/reference/spmf/README.md. Gate M1 yêu cầu so khớp toàn bộ 100%.

In [ ]:
if RUN_MINER:
    run_julia("scripts/mine.jl", "--input", "data/toy/basic.txt", "--output", "results/basic.txt",
              "--minsup", "40%", "--backend", "list")
    run_julia("scripts/compare_spmf.jl", "results/basic.txt", "data/toy/basic.expected.txt")
    run_julia("tests/runtests.jl", env={"ECLAT_TEST_MINER": "1"})
else:
    print("PENDING: Eclat basic/BitVector chưa cài đặt; chưa kiểm chứng miner.")

if RUN_SPMF:
    subprocess.run([sys.executable, "scripts/prepare_references_b.py", "--java", os.environ.get("ECLAT_JAVA", "java")], check=True)
    run_julia("tests/runtests_b.jl")
else:
    print("5 references SPMF thật của B đã có trong repo; phiên này không tạo lại. Java >=21 cần thiết nếu tạo lại.")


## M2 — BitVector và dữ liệu ứng dụng

Chỉ mở sau gate correctness M1. B điền data/manifest.csv và download cells
từ snapshot công khai thật (ưu tiên HF/Kaggle), kiểm hash và schema. Code chuẩn bị
phải viết trong notebook hoặc import helpers Julia của repo.

Groceries: một basket → transaction, mapping tên ổn định; miner nhóm → rules
support/confidence/lift → top-10 theo lift và giải thích 3 luật.

DNA: UCI Splice → 12 cặp (position,base)/mẫu; nhãn EI/IE/N giữ riêng. Áp dụng
cửa sổ cố định 25:36, thống kê ambiguity và class counts. So support của pattern
2–4 item trên cả ba lớp, đếm lại candidate infrequent ở lớp khác, không gán zero.

In [ ]:
if RUN_OPTIMIZED:
    run_julia("tests/runtests.jl", env={"ECLAT_TEST_MINER": "1", "ECLAT_TEST_SPMF": "1",
                                      "ECLAT_TEST_BACKENDS": "list,bitset"})
else:
    print("PENDING M2: chưa kiểm chứng BitVector với basic/SPMF.")

if RUN_APPLICATIONS:
    raise RuntimeError("Chỉ tích hợp miner/rules sau khi A được duyệt và merge; nhánh B hiện chỉ chuẩn bị/kiểm chứng dữ liệu.")
else:
    print("Preprocessing B đã có; kết quả ứng dụng mined và rules thật chờ miner được duyệt.")


## M3 — benchmark và hình/bảng báo cáo

A: configs/experiments.toml gồm Chess, Mushroom, Retail, T10I4D100K, sáu minsup/tập.
Hoàn thiện runner gọi benchmark_worker.jl, SPMF, warm-up/repeats, timeout và OS peak RSS.
Lưu raw CSV/log và vẽ runtime/output size; memory ở minsup trung bình; scalability
Retail 10/25/50/75/100%; synthetic width tăng dần. B đối chiếu correctness benchmark.

Không trộn timer Julia read+mine với SPMF timer khác phạm vi. Allocations không phải
peak RAM. Xem docs/experiments.md. Chèn số liệu thật vào Report.tex ba chương.

In [ ]:
if RUN_BENCHMARKS:
    print((ROOT / "configs/experiments.toml").read_text(encoding="utf-8"))
    raise NotImplementedError(
        "TODO M3 / A: thêm điều phối 4 benchmark, SPMF, OS RSS, aggregation và plotting; "
        "không thay bằng số liệu điền tay."
    )
else:
    print("PENDING: chưa chạy benchmark thực; không có số đo để vẽ hoặc kết luận.")

## Final cold-run — B chủ trì, A review

Trước nộp: bỏ mọi TODO cần thực thi, bật đủ cờ, restart runtime và Run all.
Đặt FINAL_DELIVERY_VALIDATION=True để kiểm artifacts cơ bản; vẫn cần reviewer kiểm
chất lượng nội dung, đủ đồ thị, đúng metrics và public links. Report phải 10–15
trang IEEE tiếng Việt, paper gốc + ≥3 nguồn khác. README phải có public Colab URL.

In [ ]:
if FINAL_DELIVERY_VALIDATION:
    assert all([RUN_MINER, RUN_SPMF, RUN_OPTIMIZED, RUN_APPLICATIONS, RUN_BENCHMARKS]), "Chưa bật đủ pipeline."
    run_julia("tests/runtests.jl", env={"ECLAT_TEST_MINER": "1", "ECLAT_TEST_SPMF": "1",
                                      "ECLAT_TEST_BACKENDS": "list,bitset"})
    for relative in ["docs/Report.pdf", "results/applications/groceries_rules.csv",
                     "results/applications/dna_patterns.csv"]:
        assert (ROOT / relative).is_file(), f"Thiếu artifact: {relative}"
    print("Đã qua kiểm tra artifacts cơ bản; A/B tiếp tục rà checklist DELIVERY.md.")
else:
    print("TEMPLATE ONLY: chưa cold-run nghiệm thu đồ án, chưa có kết quả cuối.")